In [ ]:
import requests
import datetime
import logging
import time
from typing import Optional, Dict, Any
from dataclasses import dataclass
import statistics

@dataclass
class TrafficResult:
    score: float
    confidence: float
    sources_used: int
    timestamp: str
    error: Optional[str] = None

class ProductionTrafficCalculator:
    def __init__(self, google_key: str, here_key: str = None, tomtom_key: str = None,
                 openweather_key: str = None, event_key: str = None):
        
        self.api_keys = {
            'google': google_key,
            'here': here_key,
            'tomtom': tomtom_key,
            'openweather': openweather_key,
            'event': event_key
        }
        
        # Configuration
        self.config = {
            'timeout': (3.05, 10),  # Connect, read timeouts
            'max_retries': 3,
            'retry_backoff': 1.5,
            'cache_ttl': 300  # 5 minutes
        }
        
        self.logger = self._setup_logging()
        self.session = self._setup_session()
        
    def _setup_logging(self) -> logging.Logger:
        logger = logging.getLogger(__name__)
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
            )
            handler.setFormatter(formatter)
            logger.addHandler(handler)
            logger.setLevel(logging.INFO)
        return logger

    def _setup_session(self) -> requests.Session:
        session = requests.Session()
        session.headers.update({
            'User-Agent': 'TrafficCalculator/1.0',
            'Accept': 'application/json'
        })
        return session

    def safe_api_call(self, url: str, params: Dict, service_name: str) -> Optional[Dict]:
        """Robust API call with retry logic and error handling"""
        for attempt in range(self.config['max_retries']):
            try:
                self.logger.debug(f"Attempt {attempt + 1} for {service_name}")
                
                response = self.session.get(
                    url, 
                    params=params, 
                    timeout=self.config['timeout']
                )
                
                if response.status_code == 429:
                    backoff = (self.config['retry_backoff'] ** attempt)
                    self.logger.warning(f"Rate limited, backing off {backoff}s")
                    time.sleep(backoff)
                    continue
                    
                response.raise_for_status()
                return response.json()
                
            except requests.exceptions.Timeout:
                self.logger.warning(f"Timeout calling {service_name}")
                if attempt == self.config['max_retries'] - 1:
                    self.logger.error(f"All retries failed for {service_name}")
                    return None
                    
            except requests.exceptions.RequestException as e:
                self.logger.error(f"Request failed for {service_name}: {e}")
                return None
                
        return None

    def get_google_travel_time(self, origin: tuple, destination: tuple) -> Optional[Dict]:
        """Google Distance Matrix API with proper error handling"""
        if not self.api_keys['google']:
            return None
            
        url = "https://maps.googleapis.com/maps/api/distancematrix/json"
        params = {
            "origins": f"{origin[0]},{origin[1]}",
            "destinations": f"{destination[0]},{destination[1]}",
            "departure_time": "now",
            "traffic_model": "best_guess",
            "key": self.api_keys['google']
        }
        
        return self.safe_api_call(url, params, "google_traffic")

    def get_weather_impact(self, lat: float, lon: float) -> float:
        """Calculate weather impact on traffic (0-1 scale)"""
        if not self.api_keys['openweather']:
            return 0.0
            
        url = "http://api.openweathermap.org/data/2.5/weather"
        params = {
            "lat": lat, "lon": lon, 
            "appid": self.api_keys['openweather'], 
            "units": "metric"
        }
        
        data = self.safe_api_call(url, params, "weather")
        if not data or 'weather' not in data:
            return 0.0
            
        weather_main = data['weather'][0]['main'].lower()
        
        # Weather impact weights
        impacts = {
            'thunderstorm': 0.8,
            'snow': 0.7, 
            'rain': 0.5,
            'drizzle': 0.3,
            'clear': 0.0,
            'clouds': 0.1
        }
        
        return impacts.get(weather_main, 0.2)

    def get_event_impact(self, lat: float, lon: float) -> float:
        """Calculate event impact on traffic"""
        if not self.api_keys['event']:
            return 0.0
            
        url = "https://app.ticketmaster.com/discovery/v2/events.json"
        params = {
            "latlong": f"{lat},{lon}", 
            "radius": 10,  # 10 miles
            "apikey": self.api_keys['event'],
            "size": 5  # Limit results
        }
        
        data = self.safe_api_call(url, params, "events")
        if not data or '_embedded' not in data:
            return 0.0
            
        event_count = len(data['_embedded']['events'])
        return min(1.0, event_count * 0.2)  # Scale impact

    def calculate_congestion_score(self, travel_data: Dict) -> Optional[float]:
        """Calculate congestion score from travel data"""
        try:
            element = travel_data['rows'][0]['elements'][0]
            
            if element.get('status') != 'OK':
                return None
                
            if 'duration_in_traffic' not in element:
                return None
                
            base_time = element['duration']['value']  # seconds
            traffic_time = element['duration_in_traffic']['value']  # seconds
            
            if base_time <= 0:
                return None
                
            congestion_ratio = traffic_time / base_time
            return min(3.0, congestion_ratio)  # Cap at 3x congestion
            
        except (KeyError, IndexError, TypeError) as e:
            self.logger.error(f"Error parsing travel data: {e}")
            return None

    def calculate_hybrid_score(self, origin: tuple, destination: tuple) -> TrafficResult:
        """Production-ready hybrid traffic calculation"""
        start_time = time.time()
        scores = []
        sources_used = 0
        errors = []

        try:
            # 1. Google Traffic (Primary)
            travel_data = self.get_google_travel_time(origin, destination)
            if travel_data:
                congestion_score = self.calculate_congestion_score(travel_data)
                if congestion_score:
                    scores.append(congestion_score * 50)  # Convert to 0-100 scale
                    sources_used += 1
                else:
                    errors.append("Google traffic data unavailable")
            else:
                errors.append("Google API call failed")

            # 2. Weather Impact
            weather_impact = self.get_weather_impact(*origin)
            if weather_impact > 0:
                scores.append(weather_impact * 30)  # Weather can add up to 30 points
                sources_used += 1

            # 3. Event Impact
            event_impact = self.get_event_impact(*origin)
            if event_impact > 0:
                scores.append(event_impact * 20)  # Events can add up to 20 points
                sources_used += 1

            # Calculate final score
            if scores:
                final_score = min(100, sum(scores))
                confidence = min(100, 60 + (sources_used * 15))  # Base 60% + 15% per source
            else:
                final_score = 0
                confidence = 0
                errors.append("No data sources available")

            calculation_time = round(time.time() - start_time, 2)
            self.logger.info(f"Calculation completed in {calculation_time}s - Score: {final_score}")

            return TrafficResult(
                score=final_score,
                confidence=confidence,
                sources_used=sources_used,
                timestamp=datetime.datetime.now().isoformat(),
                error="; ".join(errors) if errors else None
            )

        except Exception as e:
            self.logger.error(f"Unexpected error in hybrid calculation: {e}")
            return TrafficResult(
                score=0,
                confidence=0,
                sources_used=0,
                timestamp=datetime.datetime.now().isoformat(),
                error=str(e)
            )

# Usage example
def main():
    calculator = ProductionTrafficCalculator(
        google_key="your_google_key",
        openweather_key="your_weather_key",
        event_key="your_event_key"
    )
    
    result = calculator.calculate_hybrid_score(
        origin=(40.7580, -73.9855),  # Times Square
        destination=(40.7484, -73.9857)  # Empire State Building
    )
    
    print(f"Score: {result.score}/100")
    print(f"Confidence: {result.confidence}%")
    print(f"Sources: {result.sources_used}")
    if result.error:
        print(f"Errors: {result.error}")

if __name__ == "__main__":
    main()


=== Times Square, NYC ===
Fetching Google Places data...
Google Places API error: REQUEST_DENIED
Fetching real-time traffic data...
Estimating population density...
Google Places API error: REQUEST_DENIED
Fetching road network data...
Traffic Score: 0.4/100
Real-time Traffic Level: 1.0x
POIs Found: 0
Calculation Time: 6.17s
Confidence: medium

=== Silicon Valley ===
Fetching Google Places data...
Google Places API error: REQUEST_DENIED
Fetching real-time traffic data...
Estimating population density...
Google Places API error: REQUEST_DENIED
Fetching road network data...
Traffic Score: 0.1/100
Real-time Traffic Level: 1.0x
POIs Found: 0
Calculation Time: 12.38s
Confidence: medium

=== Downtown Chicago ===
Fetching Google Places data...
Google Places API error: REQUEST_DENIED
Fetching real-time traffic data...
Estimating population density...
Google Places API error: REQUEST_DENIED
Fetching road network data...
Traffic Score: 0.4/100
Real-time Traffic Level: 1.0x
POIs Found: 0
Calculat